In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv('/content/drive/MyDrive/MarketplaceApps/CensusProjectFiles/pep_county_merged_master.csv')

/tmp/ipykernel_20842/3717476829.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/content/drive/MyDrive/MarketplaceApps/CensusProjectFiles/pep_county_merged_master.csv')


In [ ]:
censusData = data.copy()

In [ ]:
censusData.head()

,NAME,POP,DENSITY,SURFACE_AREA,DATE_CODE,DATE_DESC,state,county,PERIOD_CODE,PERIOD_DESC,...,INTERNATIONALMIG,DOMESTICMIG,NETMIG,RESIDUAL,RBIRTH,RDEATH,RNATURALCHG,RINTERNATIONALMIG,RDOMESTICMIG,RNETMIG
0,"Autauga County, Alabama",43751,NaN,NaN,0,4/1/2000 to 6/30/2000,1,1,0,4/1/2000 to 6/30/2000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Autauga County, Alabama",44509,NaN,NaN,-9,7/1/2001 to 6/30/2002,1,1,-9,7/1/2001 to 6/30/2002,...,NaN,NaN,318.0,NaN,14.2668,7.9534,6.3133,NaN,NaN,7.1446
2,"Autauga County, Alabama",45207,NaN,NaN,-8,7/1/2002 to 6/30/2003,1,1,-8,7/1/2002 to 6/30/2003,...,NaN,NaN,507.0,NaN,13.1174,8.1182,4.9992,NaN,NaN,11.2151
3,"Autauga County, Alabama",45762,NaN,NaN,-7,7/1/2003 to 6/30/2004,1,1,-7,7/1/2003 to 6/30/2004,...,NaN,NaN,384.0,NaN,13.3954,8.7627,4.6327,NaN,NaN,8.3912
4,"Autauga County, Alabama",46948,NaN,NaN,-6,7/1/2004 to 6/30/2005,1,1,-6,7/1/2004 to 6/30/2005,...,NaN,NaN,1001.0,NaN,13.5256,8.4349,5.0907,NaN,NaN,21.3215


In [ ]:
censusData.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78570 entries, 0 to 78569
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   NAME               78570 non-null  object 
 1   POP                78570 non-null  int64  
 2   DENSITY            58575 non-null  float64
 3   SURFACE_AREA       58630 non-null  object 
 4   DATE_CODE          78570 non-null  int64  
 5   DATE_DESC          78570 non-null  object 
 6   state              78570 non-null  int64  
 7   county             78570 non-null  int64  
 8   PERIOD_CODE        78570 non-null  int64  
 9   PERIOD_DESC        78570 non-null  object 
 10  BIRTHS             75427 non-null  float64
 11  DEATHS             75427 non-null  float64
 12  NATURALINC         75427 non-null  float64
 13  INTERNATIONALMIG   47140 non-null  float64
 14  DOMESTICMIG        47140 non-null  float64
 15  NETMIG             75427 non-null  float64
 16  RESIDUAL           471

In [ ]:
df = pd.DataFrame(censusData)

####Clean out NAs and define types so that we can work on those columns

In [ ]:
df['SURFACE_AREA'] = df['SURFACE_AREA'].fillna('')
df['DENSITY'] = df['DENSITY'].fillna('')

In [ ]:
df['SURFACE_AREA'] = pd.to_numeric(df['SURFACE_AREA'], errors='coerce')
df['DENSITY'] = pd.to_numeric(df['DENSITY'], errors='coerce')

In [ ]:
df["SURFACE_AREA"] = df["SURFACE_AREA"].astype(float)

####Create a new DF with the average Surface areas for each county

In [ ]:
sa = df.groupby(['state','county'])['SURFACE_AREA'].mean()

In [ ]:
sa.head()

state  county
1      1          594.44
       3         1589.82
       5          885.01
       7          622.46
       9          644.83
Name: SURFACE_AREA, dtype: float64

In [ ]:
sa = sa.to_frame()

In [ ]:
#Rename column to avoid ambiguity after join
sa = sa.rename(columns={'SURFACE_AREA': 'SURFACE_AREA_Avg'})
sa.head()

SURFACE_AREA_Avg
state county                  
1     1                 594.44
      3                1589.82
      5                 885.01
      7                 622.46
      9                 644.83

####Now join the two data frames on state and county, calculate the density and drop unnecessary columns

In [ ]:
df2 = pd.merge(df, sa, on=['state', 'county'], how='inner')


In [ ]:
df2.sort_values(by='NAME')[:25]

,NAME,POP,DENSITY,SURFACE_AREA,DATE_CODE,DATE_DESC,state,county,PERIOD_CODE,PERIOD_DESC,...,DOMESTICMIG,NETMIG,RESIDUAL,RBIRTH,RDEATH,RNATURALCHG,RINTERNATIONALMIG,RDOMESTICMIG,RNETMIG,SURFACE_AREA_Avg
31440,"Abbeville County, South Carolina",24262,49.370500,491.43,2,7/1/2020 population estimate,45,1,11,4/1/2020 to 6/30/2020,...,-21.0,-21.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,491.43
31438,"Abbeville County, South Carolina",24567,49.991156,491.43,10,7/1/2017 population estimate,45,1,9,7/1/2017 to 6/30/2018,...,54.0,43.0,0.0,9.643162,10.578997,NaN,-0.447573,2.197176,1.749603,491.43
31439,"Abbeville County, South Carolina",24587,50.031854,491.43,11,7/1/2018 population estimate,45,1,10,7/1/2018 to 6/30/2019,...,-3.0,-14.0,-2.0,9.406686,11.198436,NaN,-0.447937,-0.122165,-0.570102,491.43
31437,"Abbeville County, South Carolina",24657,50.174296,491.43,9,7/1/2016 population estimate,45,1,8,7/1/2016 to 6/30/2017,...,13.0,6.0,-2.0,8.816837,12.636112,NaN,-0.284414,0.528198,0.243784,491.43
31436,"Abbeville County, South Carolina",24796,50.457146,491.43,8,7/1/2015 population estimate,45,1,7,7/1/2015 to 6/30/2016,...,-46.0,-53.0,-2.0,9.180434,12.577599,NaN,-0.283097,-1.860352,-2.143449,491.43
31435,"Abbeville County, South Carolina",24795,50.455111,491.43,7,7/1/2014 population estimate,45,1,6,7/1/2014 to 6/30/2015,...,80.0,73.0,-2.0,9.961485,12.784578,NaN,-0.282309,3.226392,2.944083,491.43
31434,"Abbeville County, South Carolina",24899,50.666740,491.43,6,7/1/2013 population estimate,45,1,5,7/1/2013 to 6/30/2014,...,-76.0,-82.0,-3.0,9.739606,10.504286,NaN,-0.241478,-3.058719,-3.300197,491.43
31432,"Abbeville County, South Carolina",25081,51.037090,491.43,4,7/1/2011 population estimate,45,1,3,7/1/2011 to 6/30/2012,...,-106.0,-88.0,-2.0,11.177645,10.059880,NaN,0.718563,-4.231537,-3.512974,491.43
31431,"Abbeville County, South Carolina",25328,51.539708,491.43,3,7/1/2010 population estimate,45,1,2,7/1/2010 to 6/30/2011,...,-192.0,-186.0,0.0,9.085679,11.505882,NaN,0.238053,-7.617687,-7.379635,491.43
31430,"Abbeville County, South Carolina",25416,51.718778,491.43,2,4/1/2010 population estimates base,45,1,1,4/1/2010 to 6/30/2010,...,-67.0,-65.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,491.43


In [ ]:
df2['DENSITY'] = df2['POP']/df2['SURFACE_AREA_Avg']

In [ ]:
df2['SURFACE_AREA'] = df2['SURFACE_AREA_Avg']


In [ ]:
df2 = df2.drop(columns=['SURFACE_AREA_Avg'])

In [51]:
df2.head()

,NAME,POP,DENSITY,SURFACE_AREA,DATE_CODE,DATE_DESC,state,county,PERIOD_CODE,PERIOD_DESC,...,INTERNATIONALMIG,DOMESTICMIG,NETMIG,RESIDUAL,RBIRTH,RDEATH,RNATURALCHG,RINTERNATIONALMIG,RDOMESTICMIG,RNETMIG
0,"Autauga County, Alabama",43751,73.600363,594.44,0,4/1/2000 to 6/30/2000,1,1,0,4/1/2000 to 6/30/2000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Autauga County, Alabama",44509,74.875513,594.44,-9,7/1/2001 to 6/30/2002,1,1,-9,7/1/2001 to 6/30/2002,...,NaN,NaN,318.0,NaN,14.2668,7.9534,6.3133,NaN,NaN,7.1446
2,"Autauga County, Alabama",45207,76.049727,594.44,-8,7/1/2002 to 6/30/2003,1,1,-8,7/1/2002 to 6/30/2003,...,NaN,NaN,507.0,NaN,13.1174,8.1182,4.9992,NaN,NaN,11.2151
3,"Autauga County, Alabama",45762,76.983379,594.44,-7,7/1/2003 to 6/30/2004,1,1,-7,7/1/2003 to 6/30/2004,...,NaN,NaN,384.0,NaN,13.3954,8.7627,4.6327,NaN,NaN,8.3912
4,"Autauga County, Alabama",46948,78.978534,594.44,-6,7/1/2004 to 6/30/2005,1,1,-6,7/1/2004 to 6/30/2005,...,NaN,NaN,1001.0,NaN,13.5256,8.4349,5.0907,NaN,NaN,21.3215


In [54]:
#Confirm that we have all of the densities we need before exporting the new DF back to Drive
count = len(df2[df2['DENSITY'] >0])
print(count)

78485


In [55]:
df2.to_csv('/content/drive/MyDrive/MarketplaceApps/CensusProjectFiles/pep_county_merged_master_2.csv',index=False)